# Étude de cas : ventes et livraisons

Une entreprise vend des produits en ligne (épicerie, frais, boissons, hygiène, maison, électronique, livres, bricolage, collection) et les livre elle-même, depuis ses entrepôts de la région parisienne, avec sa propre flotte de véhicules.

### Les fichiers (`/work/data`)

| Fichier | Contenu |
|---|---|
| `produits.csv` | le catalogue : `produit_id, nom, categorie, sous_categorie, prix, poids_kg` |
| `commandes.csv` | le journal des commandes, une ligne par événement : `commande_id, client_id, horodatage, evenement, livreur_id, detail` |
| `vehicules/<vehicule>/<AAAA-MM>.csv` | la trace GPS de chaque véhicule, **un point par minute** : `timestamp, livreur_id, lon, lat, vitesse_kmh, dans_vehicule, charge_kg` |
| `communes.csv` | les communes de Paris et de la petite couronne et les 20 arrondissements de Paris, avec leur centre : `commune, code_postal, lon, lat` |
| `communes.geojson` | les contours des mêmes communes (propriétés `commune`, `code_postal`) |

### Le journal des commandes

Une commande apparaît plusieurs fois : une ligne par étape.

| `evenement` | Sens | `livreur_id` | `detail` |
|---|---|---|---|
| `commande` | le client passe commande | — | les produits : `P0201x1\|P0033x4` (produit `x` quantité) |
| `chargee` | la commande est chargée dans un véhicule | le livreur | — |
| `livree` | le client signe ; horodatage à la seconde | le livreur | — |
| `echec` | la livraison n'a pas pu se faire | le livreur | le motif |
| `reportee` | chargée mais pas tentée : rapportée à l'entrepôt en fin de service | le livreur | — |
| `retour` | la commande est abandonnée et retournée à l'entrepôt | le livreur | — |

### Les traces GPS

Un livreur conduit un seul véhicule par jour. `dans_vehicule` vaut `false` quand le livreur est descendu du véhicule ; `charge_kg` est le poids transporté.

### Les services

| Adresse | Service |
|---|---|
| http://localhost:8888 | JupyterLab |
| http://localhost:8080 | Master Spark |
| http://localhost:4040 | Spark UI de la session ouverte |
| http://localhost:18080 | Spark History |
| `mongodb://mongo:27017` | MongoDB |

## 1) Charger la donnée

1.1) La session Spark et les fonctions fournies.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("livraisons")
         .master("spark://spark-master:7077")
         .getOrCreate())
DATA = "/work/data"
RESULTATS = "/work/resultats"


def haversine(lon1, lat1, lon2, lat2):
    """Distance en km entre deux points : noms de colonnes ou colonnes, en degrés."""
    lon1, lat1, lon2, lat2 = [F.col(c) if isinstance(c, str) else c for c in (lon1, lat1, lon2, lat2)]
    dlon = F.radians(lon2 - lon1)
    dlat = F.radians(lat2 - lat1)
    a = F.sin(dlat / 2) ** 2 + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(dlon / 2) ** 2
    return 2 * 6371.0 * F.asin(F.sqrt(a))

11:42:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


1.2) Définissez un schéma explicite (`StructType`) pour chacun des 4 fichiers CSV, puis chargez `produits.csv`, `commandes.csv`, `communes.csv` et **tous** les fichiers GPS en une seule lecture (`vehicules/*/*.csv`). Les horodatages doivent être de type `timestamp`, `dans_vehicule` de type `boolean`, les nombres de type `double`.

Affichez le nombre de lignes de chaque DataFrame.



In [2]:
schema_produits = T.StructType([
    T.StructField("produit_id", T.StringType()),
    T.StructField("nom", T.StringType()),
    T.StructField("categorie", T.StringType()),
    T.StructField("sous_categorie", T.StringType()),
    T.StructField("prix", T.DoubleType()),
    T.StructField("poids_kg", T.DoubleType()),
])
schema_journal = T.StructType([
    T.StructField("commande_id", T.StringType()),
    T.StructField("client_id", T.StringType()),
    T.StructField("horodatage", T.TimestampType()),
    T.StructField("evenement", T.StringType()),
    T.StructField("livreur_id", T.StringType()),
    T.StructField("detail", T.StringType()),
])
schema_communes = T.StructType([
    T.StructField("commune", T.StringType()),
    T.StructField("code_postal", T.StringType()),
    T.StructField("lon", T.DoubleType()),
    T.StructField("lat", T.DoubleType()),
])
schema_gps = T.StructType([
    T.StructField("timestamp", T.TimestampType()),
    T.StructField("livreur_id", T.StringType()),
    T.StructField("lon", T.DoubleType()),
    T.StructField("lat", T.DoubleType()),
    T.StructField("vitesse_kmh", T.DoubleType()),
    T.StructField("dans_vehicule", T.BooleanType()),
    T.StructField("charge_kg", T.DoubleType()),
])

produits = spark.read.csv(f"{DATA}/produits.csv", header=True, schema=schema_produits)
journal  = spark.read.csv(f"{DATA}/commandes.csv", header=True, schema=schema_journal)
communes = spark.read.csv(f"{DATA}/communes.csv", header=True, schema=schema_communes)
gps      = spark.read.csv(f"{DATA}/vehicules/*/*.csv", header=True, schema=schema_gps)

print("produits :", produits.count())
print("journal  :", journal.count())
print("communes :", communes.count())
print("gps      :", gps.count())


produits : 233


journal  : 846535
communes : 142


gps      : 5174905


1.3) L'identifiant du véhicule n'est pas une colonne des fichiers GPS : il est dans le chemin du fichier. Ajoutez une colonne `vehicule_id` (fonction `F.input_file_name()`), et une colonne `jour` (la date du point).

In [3]:
gps = (gps
       .withColumn("vehicule_id", F.regexp_extract(F.input_file_name(), r"vehicules/([^/]+)/", 1))
       .withColumn("jour", F.to_date("timestamp")))
gps.show(5)


+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+
|          timestamp|livreur_id|     lon|      lat|vitesse_kmh|dans_vehicule|charge_kg|vehicule_id|      jour|
+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+
|2025-10-01 07:40:00|      L001|2.277083|48.936735|        0.0|        false|   2581.6|        V18|2025-10-01|
|2025-10-01 07:41:00|      L001|2.277083|48.936735|        0.0|        false|   2581.6|        V18|2025-10-01|
|2025-10-01 07:42:00|      L001|2.277084|48.936734|        0.0|        false|   2581.6|        V18|2025-10-01|
|2025-10-01 07:43:00|      L001|2.277083|48.936734|        0.0|        false|   2581.6|        V18|2025-10-01|
|2025-10-01 07:44:00|      L001|2.277084|48.936733|        0.0|        false|   2581.6|        V18|2025-10-01|
+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+
o

## 2) Vérifier la donnée

2.1) Combien y a-t-il de commandes, de clients, de livreurs et de véhicules ?

In [4]:
journal.agg(F.countDistinct("commande_id").alias("nb_commandes"),
            F.countDistinct("client_id").alias("nb_clients"),
            F.countDistinct("livreur_id").alias("nb_livreurs")).show()
gps.agg(F.countDistinct("vehicule_id").alias("nb_vehicules")).show()


+------------+----------+-----------+
|nb_commandes|nb_clients|nb_livreurs|
+------------+----------+-----------+
|      271617|      8000|         90|
+------------+----------+-----------+



+------------+
|nb_vehicules|
+------------+
|          72|
+------------+



2.2) Comptez les événements du journal par type.

In [6]:
journal.groupBy("evenement").count().orderBy(F.desc("count")).show()

+---------+------+
|evenement| count|
+---------+------+
|  chargee|286862|
| commande|271617|
|   livree|268682|
|    echec| 11220|
| reportee|  6960|
|   retour|  1194|
+---------+------+



2.3) Pour chaque commande, comptez ses événements `chargee`. Combien de commandes ont été chargées 0, 1, 2, 3… fois ?

In [12]:
nb_charge = journal.groupBy("commande_id").agg(
    F.sum(F.when(F.col("evenement") == "chargee", 1).otherwise(0)).alias("nb_chargee"))

nb_charge.groupBy("nb_chargee").count().orderBy("nb_chargee").show()
print("Total commandes  :", nb_charge.count())                                   # 271617 (= 2.1)
print("Total chargements :", nb_charge.agg(F.sum("nb_chargee")).first()[0])      # 286862 (= 2.2)



+----------+------+
|nb_chargee| count|
+----------+------+
|         0|  1637|
|         1|254229|
|         2| 14673|
|         3|  1026|
|         4|    51|
|         5|     1|
+----------+------+



Total commandes  : 271617


Total chargements : 286862


ici j'ai  voulu garder les commandes jmais chargées. si j'avais fait un filter par evnement chargé, les non chargées auraient disparu

2.4) Pour chaque événement `chargee`, trouvez l'événement qui le précède dans la vie de la même commande. Comptez les `chargee` par événement précédent.

In [13]:
w_cmd = Window.partitionBy("commande_id").orderBy("horodatage")

journal_prec = journal.withColumn("evenement_precedent", F.lag("evenement").over(w_cmd))

(journal_prec.filter(F.col("evenement") == "chargee")
    .groupBy("evenement_precedent").count()
    .orderBy(F.desc("count")).show())
print("Total :", journal_prec.filter(F.col("evenement") == "chargee").count())   # 286862 (= 2.2)

## la partition range ensemble les evenement d'une meme commande, et order by va trié dans l'orde chronologique.. le f.lag va chercher la valeur de la ligne juste avant.

+-------------------+------+
|evenement_precedent| count|
+-------------------+------+
|           commande|269980|
|              echec|  9950|
|           reportee|  6932|
+-------------------+------+



Total : 286862


*Un rechargement suit toujours un `echec` (nouvelle tentative) ou un `reportee` (commande rapportée en fin de service) : il y a plus de `chargee` que de `commande`.*

2.5) Au 31/12/2025, combien de commandes ne sont pas livrées ? Séparez les commandes abandonnées (événement `retour`) des commandes encore en cours.

In [9]:
statut = journal.groupBy("commande_id").agg(
    F.max(F.when(F.col("evenement") == "livree", 1).otherwise(0)).alias("est_livree"),
    F.max(F.when(F.col("evenement") == "retour", 1).otherwise(0)).alias("est_retour"))

non_livrees = statut.filter("est_livree = 0")

print("Commandes non livrées au 31/12 :", non_livrees.count())
print("  - abandonnées (retour)        :", non_livrees.filter("est_retour = 1").count())
print("  - encore en cours             :", non_livrees.filter("est_retour = 0").count())
## ici on cherhce si c'est arrivé au moins une fois avec max (est_retour = 1 correspond aux commandes abandonnées)

Commandes non livrées au 31/12 : 2935


  - abandonnées (retour)        : 1194


  - encore en cours             : 1741


## 3) Les lignes de commande

3.1) Les produits d'une commande sont dans la colonne `detail` de l'événement `commande`. Construisez la table `lignes` : une ligne par produit commandé, avec `commande_id`, `client_id`, `horodatage`, `produit_id`, `quantite`.

In [16]:
lignes_brutes = (journal.filter(F.col("evenement") == "commande")
    .withColumn("produit", F.explode(F.split("detail", r"\|")))
    .withColumn("produit_id", F.split("produit", "x")[0])
    .withColumn("quantite", F.split("produit", "x")[1].cast("int"))
    .select("commande_id", "client_id", "horodatage", "produit_id", "quantite"))

lignes_brutes.filter(F.col("commande_id") == "C0000001").show(truncate=False)
print("Lignes de commande :", lignes_brutes.count())


+-----------+---------+-------------------+----------+--------+
|commande_id|client_id|horodatage         |produit_id|quantite|
+-----------+---------+-------------------+----------+--------+
|C0000001   |CL05658  |2025-01-01 00:03:44|P0037     |5       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0063     |1       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0019     |2       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0067     |1       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0069     |2       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0089     |3       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0061     |1       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0055     |2       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0009     |3       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0076     |1       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0107     |1       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0005     |2       |
|C0000001   |CL05658  |2025-01-01 00:03:

Lignes de commande : 4779049


3.2) Ajoutez à `lignes` les colonnes du catalogue et le `montant` de chaque ligne. Construisez la table `commandes` : une ligne par commande, avec `montant`, `poids_kg` et `nb_lignes`.

In [17]:
lignes = (lignes_brutes.join(produits, "produit_id", "left")
          .withColumn("montant", F.round(F.col("prix") * F.col("quantite"), 2))
          .cache())

commandes = (lignes.groupBy("commande_id", "client_id", "horodatage").agg(
                 F.round(F.sum("montant"), 2).alias("montant"),
                 F.round(F.sum(F.col("poids_kg") * F.col("quantite")), 2).alias("poids_kg"),
                 F.count("*").alias("nb_lignes"))
             .cache())

lignes.filter(F.col("commande_id") == "C0000001") \
      .select("commande_id", "produit_id", "nom", "categorie", "quantite", "prix", "montant").show(truncate=False)
commandes.orderBy("commande_id").show(5)

print("Lignes de commande :", lignes.count())
print("Commandes          :", commandes.count(), "(attendu 271617, cf. 2.1)")
print("Produits sans prix :", lignes.filter(F.col("prix").isNull()).count(), "(attendu 0)")


+-----------+----------+----------------------------+-----------------+--------+----+-------+
|commande_id|produit_id|nom                         |categorie        |quantite|prix|montant|
+-----------+----------+----------------------------+-----------------+--------+----+-------+
|C0000001   |P0037     |Cookies chocolat 300 g      |Épicerie         |5       |2.6 |13.0   |
|C0000001   |P0063     |Lardons fumés 200 g         |Frais            |1       |2.4 |2.4    |
|C0000001   |P0019     |Capsules café x30           |Épicerie         |2       |8.5 |17.0   |
|C0000001   |P0067     |Bananes 1 kg                |Frais            |1       |1.9 |1.9    |
|C0000001   |P0069     |Tomates grappe 1 kg         |Frais            |2       |3.4 |6.8    |
|C0000001   |P0089     |Sirop de grenadine 75 cl    |Boissons         |3       |2.6 |7.8    |
|C0000001   |P0061     |Jambon blanc x4             |Frais            |1       |3.2 |3.2    |
|C0000001   |P0055     |Emmental râpé 200 g         |Frais  

+-----------+---------+-------------------+-------+--------+---------+
|commande_id|client_id|         horodatage|montant|poids_kg|nb_lignes|
+-----------+---------+-------------------+-------+--------+---------+
|   C0000001|  CL05658|2025-01-01 00:03:44| 103.65|   21.61|       19|
|   C0000002|  CL02895|2025-01-01 00:06:58|   91.3|    18.7|       15|
|   C0000003|  CL03531|2025-01-01 00:18:34|   46.0|    15.9|        9|
|   C0000004|  CL00699|2025-01-01 00:29:59|   62.1|    5.99|        8|
|   C0000005|  CL07863|2025-01-01 00:30:28|   85.4|   23.55|       13|
+-----------+---------+-------------------+-------+--------+---------+
only showing top 5 rows

Lignes de commande : 4779049


Commandes          : 271617 (attendu 271617, cf. 2.1)
Produits sans prix : 0 (attendu 0)


## 4) Les entrepôts

4.1) Pour chaque véhicule et chaque jour, gardez le premier et le dernier point GPS. Arrondissez leurs coordonnées à 0,01° et comptez les journées par position arrondie. Combien d'entrepôts y a-t-il ?

In [18]:
jours = gps.groupBy("vehicule_id", "jour").agg(
    F.min_by("lon", "timestamp").alias("lon_debut"),
    F.min_by("lat", "timestamp").alias("lat_debut"),
    F.max_by("lon", "timestamp").alias("lon_fin"),
    F.max_by("lat", "timestamp").alias("lat_fin")).cache()

debuts = jours.select(F.round("lon_debut", 2).alias("lon"), F.round("lat_debut", 2).alias("lat"))
fins   = jours.select(F.round("lon_fin", 2).alias("lon"), F.round("lat_fin", 2).alias("lat"))

positions = (debuts.union(fins)
             .groupBy("lon", "lat").agg(F.count("*").alias("nb_journees"))
             .orderBy(F.desc("nb_journees")))
positions.show()

nb_jours = jours.count()
print("Journées véhicule            :", nb_jours)
print("Points début + fin           :", nb_jours * 2)
print("Points sur ces positions     :", positions.agg(F.sum("nb_journees")).first()[0])
print("Nombre d'entrepôts           :", positions.count())


+----+-----+-----------+
| lon|  lat|nb_journees|
+----+-----+-----------+
|2.45|48.91|       8434|
|2.28|48.94|       6416|
|2.35|48.75|       5934|
+----+-----+-----------+



Journées véhicule            : 10392
Points début + fin           : 20784


Points sur ces positions     : 20784


Nombre d'entrepôts           : 3


Chaque jour, les véhicules partent et reviennent toujours à l'une des 3 mêmes positions : il y a donc 3 entrepôts.

4.2) Construisez la table `entrepots` (`entrepot`, `lon`, `lat`) : `E1`, `E2`… par longitude croissante, position = moyenne des premiers points. Construisez la table `vehicule_entrepot` (`vehicule_id`, `entrepot`) et comptez les véhicules par entrepôt.

In [20]:
# 1) Les entrepôts : position moyenne des premiers points, numérotés de l'ouest vers l'est
entrepots = (jours
    .withColumn("lon_r", F.round("lon_debut", 2))
    .withColumn("lat_r", F.round("lat_debut", 2))
    .groupBy("lon_r", "lat_r")
    .agg(F.round(F.avg("lon_debut"), 5).alias("lon"),
         F.round(F.avg("lat_debut"), 5).alias("lat")))
entrepots = entrepots.withColumn("entrepot",
                F.concat(F.lit("E"), F.row_number().over(Window.orderBy("lon")))).cache()

# 2) Le véhicule -> son entrepôt (celui de ses départs du matin)
vehicule_entrepot = (jours
    .withColumn("lon_r", F.round("lon_debut", 2))
    .withColumn("lat_r", F.round("lat_debut", 2))
    .join(entrepots, ["lon_r", "lat_r"])
    .select("vehicule_id", "entrepot").distinct().cache())

(jours.join(vehicule_entrepot, "vehicule_id")
    .groupBy("entrepot")
    .agg(F.countDistinct("vehicule_id").alias("nb_vehicules"),
         F.count("*").alias("nb_journees"))
    .withColumn("journees_par_vehicule", F.round(F.col("nb_journees") / F.col("nb_vehicules"), 0))
    .orderBy("entrepot").show())


entrepots.select("entrepot", "lon", "lat").orderBy("entrepot").show()
vehicule_entrepot.groupBy("entrepot").agg(F.count("*").alias("nb_vehicules")).orderBy("entrepot").show()

print("Véhicules rattachés         :", vehicule_entrepot.count(), "(attendu 72, cf. 2.1)")
print("Véhicules différents        :", vehicule_entrepot.select("vehicule_id").distinct().count())


14:05:25 WARN CacheManager: Asked to cache already cached data.
14:05:25 WARN CacheManager: Asked to cache already cached data.


+--------+------------+-----------+---------------------+
|entrepot|nb_vehicules|nb_journees|journees_par_vehicule|
+--------+------------+-----------+---------------------+
|      E1|          24|       3208|                134.0|
|      E2|          24|       2967|                124.0|
|      E3|          24|       4217|                176.0|
+--------+------------+-----------+---------------------+

+--------+-------+--------+
|entrepot|    lon|     lat|
+--------+-------+--------+
|      E1|2.27707|48.93671|
|      E2|2.35151|48.74705|
|      E3|2.44896| 48.9067|
+--------+-------+--------+



+--------+------------+
|entrepot|nb_vehicules|
+--------+------------+
|      E1|          24|
|      E2|          24|
|      E3|          24|
+--------+------------+



Véhicules rattachés         : 72 (attendu 72, cf. 2.1)


Véhicules différents        : 72


On nomme les 3 entrepôts E1, E2, E3 et on rattache chaque véhicule à l'entrepôt d'où il part le matin : il y a 24 véhicules par entrepôt. En comptant les jours où chaque véhicule a roulé, on voit que E3 est le plus actif (environ 176 jours par véhicule, contre environ 125 à 135 pour E1 et E2).

## 5) Les livraisons

On découpe la journée de chaque livreur en **livraisons** : une livraison commence quand le livreur descend du véhicule et dure jusqu'à la livraison suivante. Le chargement du matin, les retours à l'entrepôt et la pause déjeuner sont aussi des périodes à pied : la question 5.6 les distingue.

5.1) Ajoutez à `gps` une colonne `changement` : 1 si le livreur vient de descendre du véhicule (point précédent du même livreur, le même jour, `dans_vehicule = true` ; point courant `false`), 0 sinon. Le premier point de la journée vaut 0.

Affichez les lignes où `changement = 1` pour le livreur `L001` le 2025-01-06.

5.2) Ajoutez une colonne `numero_livraison` : la somme cumulée de `changement` pour chaque livreur et chaque jour. Elle vaut 0 au chargement du matin.

Pour `L001` le 2025-01-06, affichez par `numero_livraison` le premier et le dernier timestamp, et le nombre de minutes à pied.

5.3) Construisez la table `livraisons` : une ligne par (`livreur_id`, `jour`, `numero_livraison`), calculée sur les seuls points où le livreur est à pied, avec `vehicule_id`, `debut`, `fin`, `duree_min` (nombre de points) et la position moyenne `lon`, `lat`. Combien de lignes contient-elle ?

5.4) Ajoutez à chaque livraison ses signatures et ses échecs. Un événement `livree` ou `echec` appartient à une livraison quand il a le même livreur, le même jour, et un horodatage compris entre `debut` et `fin`, avec 1 minute de marge de chaque côté : les points GPS sont à la minute, les événements à la seconde.

Construisez la table `livraisons_ev` : les colonnes de `livraisons`, plus `signature` (horodatage de l'événement), `evenement`, `commande_id` et `client_id`. Une ligne par événement ; une livraison sans événement garde une ligne, avec ces 4 colonnes vides.

5.5) Combien d'événements `livree` et `echec` du journal ne sont rattachés à aucune livraison ?

5.6) Toutes les périodes à pied ne sont pas des livraisons. Ajoutez à `livraisons_ev` une colonne `type` :
- `livraison` : au moins un événement ;
- `entrepot` : sinon, position à moins de 300 m d'un entrepôt ;
- `pause` : les autres.

Pour chaque type : nombre de lignes de `livraisons`, durée moyenne, heure moyenne de début.

5.7) Une même livraison peut compter plusieurs événements : plusieurs commandes à la même adresse, remises à pied l'une après l'autre. Combien de livraisons ont 1, 2, 3… événements ? Combien concernent au moins 2 clients différents ?

5.8) Ajoutez une colonne `duree_ev_min` à chaque événement : le temps écoulé depuis l'événement précédent de la même livraison, ou depuis le début de la livraison pour le premier. Quelle est la durée moyenne d'une remise signée ? D'un échec ?

5.9) Pour chaque livreur : nombre de livraisons signées, nombre d'échecs, nombre de jours travaillés, livraisons signées par jour. Affichez les 10 livreurs qui signent le plus de livraisons par jour.

## 6) Les distances

6.1) Ajoutez à `gps` une colonne `distance_km` : la distance depuis le point précédent du même livreur, le même jour (`haversine`). Le premier point de la journée vaut 0.

*Un point par minute coupe les virages : la distance calculée sous-estime d'environ 12 % la distance réellement parcourue.*

6.2) Kilomètres parcourus sur l'année par chaque véhicule : les 10 premiers, et le total de la flotte.

6.3) Kilomètres parcourus sur l'année par chaque livreur : les 10 premiers.

6.4) Kilomètres parcourus par jour : moyenne par véhicule-jour, pour chaque entrepôt.

6.5) Distance entre deux livraisons : la somme de `distance_km` de chaque (`livreur_id`, `jour`, `numero_livraison`), c'est-à-dire le trajet qui suit la livraison jusqu'à la suivante. Moyenne et médiane, pour les livraisons de type `livraison`.

## 7) Les ventes

7.1) Chiffre d'affaires par mois (mois de la commande).

7.2) Panier moyen : montant moyen d'une commande.

7.3) Panier médian.

7.4) Top 10 des produits par chiffre d'affaires.

7.5) Top 10 des produits par quantité vendue.

7.6) Top 10 des clients par dépense sur l'année.

7.7) Top 10 des clients par nombre de commandes.

## 8) La qualité de service

8.1) Délai entre la commande et sa livraison, en jours calendaires (date de la signature − date de la commande). Combien de commandes livrées pour chaque délai ?

8.2) Part des commandes livrées le lendemain, pour chaque mois de commande.

8.3) Construisez la table `suivi` : une ligne par commande, avec `nb_tentatives` (événements `livree` + `echec`), `nb_echecs` et `issue` (`livree`, `retour` ou `en cours`). Combien de commandes pour chaque nombre de tentatives ?

8.4) Motifs d'échec : nombre d'échecs par motif.

8.5) Top 10 des clients au plus fort taux d'échec (échecs / tentatives), parmi les clients qui ont au moins 10 tentatives.

## 9) Les clients

On construit la table `clients` colonne par colonne : une ligne par client qui a passé au moins une commande.

9.1) Construisez la table `clients` : `client_id`, `nb_commandes`, `ca` (somme des montants), `panier_moyen`.

9.2) Ajoutez `premiere_commande` et `derniere_commande` (dates).

9.3) Ajoutez `lon` et `lat` : la position moyenne des livraisons où le client a signé. Combien de clients n'ont aucune position ?

9.4) Ajoutez `commune` et `code_postal` : la commune dont le centre (`communes.csv`) est le plus proche de la position du client.

9.5) Les 10 communes qui passent le plus de commandes, avec leur chiffre d'affaires.

9.6) Ajoutez `tentatives`, `echecs` et `taux_echec` (à partir de `suivi`, 8.3).

9.7) Ajoutez `categorie_dominante` (la catégorie où le client dépense le plus) et `part_dominante` (part de sa dépense dans cette catégorie).

9.8) Ajoutez `unites_par_ligne` : la quantité moyenne par ligne de commande.

9.9) Segments de dépense annuelle (`ca`) : moins de 1 000 €, 1 000 à 5 000 €, 5 000 à 20 000 €, 20 000 à 50 000 €, 50 000 € et plus. Ajoutez la colonne `segment`, puis pour chaque segment : nombre de clients, chiffre d'affaires, chiffre d'affaires moyen par client, part du chiffre d'affaires total.

9.10) Nouveaux clients par mois (mois de la première commande), en tableau et en graphique.

9.11) Pour chaque catégorie dominante : nombre de clients et chiffre d'affaires.

9.12) Clients atypiques : les 10 clients dont `unites_par_ligne` est la plus élevée, avec le rapport à la moyenne de tous les clients, leur `ca` et leur `categorie_dominante`.

## 10) Data visualisation

Échelles de couleur : blanc ou clair pour 0, foncé pour les valeurs élevées.

10.1) Heatmap jour de la semaine × heure du nombre de commandes passées.

10.2) Carte du nombre de livraisons signées par commune ou arrondissement (`communes.geojson`, carte choroplèthe).

10.3) Heatmap heure × mois de la vitesse moyenne quand le livreur est à bord et que le véhicule roule.

10.4) Treemap du chiffre d'affaires : catégorie → sous-catégorie.

## 11) Sauvegarde et modélisation

11.2) **Modélisation.** Les analyses précédentes recalculent les mêmes jointures et agrégations. Proposez des collections MongoDB qui rendent ces questions immédiates : pour chaque collection, ce que représente un document, ses champs, ses tableaux imbriqués, et les questions des parties 5 à 9 qu'elle simplifie. Écrivez votre proposition dans la cellule ci-dessous.

*Votre proposition :*